# NDR - Notebook 31: Tai lap pipeline TOP-1 (base simple-FT + LENGTH filter + DINOv2 trim)

## Nguon: archive.zip cua top-1 (6 file, chain long nhau da giai ma bang subset-test)
```
base 630  ->  length filter 495  ->  DINOv2 trim emb96 420  ->  emb955 391
(0.315/img)   (0.247)               (0.210)                    (0.196)
```
- Base = simple-ft + rescue (KHONG phai 229 prune/EWC). Recall cao, xuat conf tho.
- Tang 1 (LEVER CHINH, chua NB nao khai thac): LENGTH filter tren maxdim box.
  Ten file top-1: `le40_or_45_51` = drop box maxdim<=40 HOAC trong band [45,51].
  Geometry thuan, deterministic, KHONG calib tren VAL anti-correlated -> generalize sang test.
- Tang 2: DINOv2 trim CUOI, bao thu (sim<0.955), KHONG boost. Khac han NB25 (dung DINOv2 lam
  tin hieu chinh + boost -> density no 1.28 -> LB254).

## Ky luat
- Cell dau tang 2: DIEU TRA length-signature tren 20 poison box that vs high-conf real det.
  Neu length KHONG tach duoc trong data nay (NB27 do poison med=48 ~ real med=47) -> notebook bao ro,
  ta chi dua vao DINOv2 trim. Neu co band tach -> ap length filter theo dung band do.
- Xuat nhieu file A/B: base (anchor) / +length / +length+dino. LB la trong tai.


## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 11.2 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric core — maCADD (unit-tested)

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    # SPEC BTC: mỗi clean box khớp de-pois box IoU CAO NHẤT (>=t); các de-pois box thừa = FP.
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        pairs = sorted(((ious[i,j], i, j) for i in range(len(cb)) for j in range(len(pb)) if ious[i,j] >= t), reverse=True)
        for _, i, j in pairs:
            if i in mc or j in mp: continue
            mc.add(i); mp.add(j); d = float(cs[i]) - float(ps[j])
            if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
            else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)   # FN: clean chưa khớp
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)   # FP: de-pois box thừa
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Load data + poisoned model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Validation set

In [5]:
MARGIN = 6
poison_patches = []
for a in coco['annotations']:
    x,y,w,h = a['bbox']; stem = stem_by_id[a['image_id']]
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN)); x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(unlearn_imgs[stem][y0:y1, x0:x1].copy())
stems = list(unlearn_imgs)
def rand_free_spot(stem, ph, pw, rng, tries=50):
    pb = poison_boxes_by_stem.get(stem, np.array([[0,0,1,1]],np.float32))
    for _ in range(tries):
        x0 = rng.integers(16, IMG_W-pw-16); y0 = rng.integers(16, IMG_H-ph-16)
        if iou_matrix(np.array([[x0,y0,x0+pw,y0+ph]],np.float32), pb).max()<0.01: return int(x0),int(y0)
    return None
def paste_patch(img, patch, x0, y0, feather=3):
    ph,pw = patch.shape[:2]; reg = img[y0:y0+ph, x0:x0+pw]
    if reg.shape[:2]!=(ph,pw): return
    mask = np.ones((ph,pw,1),np.float32)
    if ph>2*feather and pw>2*feather:
        rh = np.minimum(np.arange(ph)[:,None]+1, np.arange(ph)[::-1][:,None]+1)/feather
        rw = np.minimum(np.arange(pw)[None,:]+1, np.arange(pw)[::-1][None,:]+1)/feather
        mask = np.clip(np.minimum(rh,rw),0,1)[...,None]
    img[y0:y0+ph, x0:x0+pw] = reg*(1-mask) + patch*mask
def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
def build_validation_set(n=80, seed=SEED):
    rng = np.random.default_rng(seed); val=[]
    for _ in range(n):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); streaks=[]
        for _ in range(rng.integers(1,4)):
            g2,box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2); streaks.append(box)
        pois = [list(map(float,b)) for b in poison_boxes_by_stem.get(stem, [])]
        for _ in range(rng.integers(0,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem,ph,pw,rng)
            if spot: x0,y0 = spot; paste_patch(img,patch,x0,y0); pois.append([x0+MARGIN,y0+MARGIN,x0+pw-MARGIN,y0+ph-MARGIN])
        val.append({'img':img,'streak_boxes':streaks,'poison_boxes':pois})
    return val
VAL = build_validation_set(80)
print('val images:', len(VAL), '| streaks:', sum(len(v['streak_boxes']) for v in VAL), '| poison:', sum(len(v['poison_boxes']) for v in VAL))

val images: 80 | streaks: 159 | poison: 163


## 5. Clean reference = CHỈ streak thật

In [6]:
def make_clean_reference(val, predictor):
    '''clean_ref = CHỈ streak thật (khớp detection model gốc để lấy conf tham chiếu).
    Poison KHÔNG vào ref -> mọi detection trên poison sẽ là FP.'''
    ref = {}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        rb,rs = [],[]
        sb = np.array(item['streak_boxes'],np.float32).reshape(-1,4)
        for k in range(len(sb)):
            if len(b):
                ious = iou_matrix(sb[k:k+1], b)[0]; j = ious.argmax()
                if ious[j]>=MATCH_IOU: rb.append(b[j]); rs.append(float(s[j]))
        ref[str(i)] = (np.array(rb,np.float32).reshape(-1,4), np.array(rs,np.float32))
    return ref
CLEAN_REF = make_clean_reference(VAL, poisoned_predictor)
print('clean reference (streak-only) built for', len(CLEAN_REF), 'val images')

clean reference (streak-only) built for 80 val images


## 6. Thước đo genuine + TTA

In [7]:
def eval_model_on_val(predictor, val, tta=False):
    fired=ft=0; kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor, item['img']) if tta else predict_image(predictor, item['img']))
        keep = s>SUBMIT_THRESH; b2,s2 = (b[keep], s[keep]) if len(b) else (b,s); pk[str(i)]=(b2,s2)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return dict(poison_firing_rate=fired/max(ft,1), streak_retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF, pk))

base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('POISONED (chưa xử lý):', {k: round(v,4) for k,v in base_stats.items()})
print('-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.')
print('   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.')

POISONED (chưa xử lý): {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.
   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.


In [8]:
# --- Confidence-fidelity diagnostic (proxy gan voi maCADD/clean model) ---
# maCADD phat theo |clean_score - depois_score| (A=10 bat doi xung). Firing/retention
# chi do CO/KHONG detect; cell nay do do KHOP CONFIDENCE voi clean model tren streak that,
# va residual confidence tren poison. Gan hon voi hanh vi top-1 (recovery + calibration).
def conf_fidelity(predictor, val, clean_ref, tta=False, floor=SUBMIT_THRESH):
    gaps=[]; pred_on_streak=[]; ref_on_streak=[]; missed=0; total_streak=0
    pois_res=[]; total_pois=0
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor,item['img']) if tta else predict_image(predictor,item['img']))
        keep = s>floor; b=(b[keep] if len(b) else b); s=(s[keep] if len(b) else s)
        rb,rs = clean_ref[str(i)]
        for k in range(len(rb)):
            total_streak+=1
            if len(b):
                ious=iou_matrix(rb[k:k+1],b)[0]; j=int(ious.argmax())
                if ious[j]>=MATCH_IOU:
                    gaps.append(abs(float(rs[k])-float(s[j]))); ref_on_streak.append(float(rs[k])); pred_on_streak.append(float(s[j])); continue
            gaps.append(float(rs[k])); missed+=1   # streak bi bo lo -> gap = toan bo conf ref (dong gop FN)
        pb=np.array(item['poison_boxes'],np.float32).reshape(-1,4)
        for k in range(len(pb)):
            total_pois+=1
            if len(b):
                ious=iou_matrix(pb[k:k+1],b)[0]
                if ious.max()>=MATCH_IOU: pois_res.append(float(s[int(ious.argmax())]))
    return dict(
        streak_conf_gap=float(np.mean(gaps)) if gaps else 0.0,
        streak_miss_rate=missed/max(total_streak,1),
        pred_conf_on_streak=float(np.mean(pred_on_streak)) if pred_on_streak else 0.0,
        ref_conf_on_streak=float(np.mean(ref_on_streak)) if ref_on_streak else 0.0,
        poison_residual_conf=float(np.mean(pois_res)) if pois_res else 0.0,
        poison_fire_frac=len(pois_res)/max(total_pois,1))

base_fid = conf_fidelity(poisoned_predictor, VAL, CLEAN_REF)
print('POISONED conf-fidelity:', {k:round(v,4) for k,v in base_fid.items()})
print('-> Muc tieu: streak_conf_gap THAP (khop conf clean model tren streak that),')
print('   poison_residual_conf ~0 (khong con FP poison). Ca hai giam = gan hanh vi top-1.')


POISONED conf-fidelity: {'streak_conf_gap': 0.0194, 'streak_miss_rate': 0.0, 'pred_conf_on_streak': 0.6822, 'ref_conf_on_streak': 0.6627, 'poison_residual_conf': 0.5937, 'poison_fire_frac': 0.9693}
-> Muc tieu: streak_conf_gap THAP (khop conf clean model tren streak that),
   poison_residual_conf ~0 (khong con FP poison). Ca hai giam = gan hanh vi top-1.


In [9]:
D4 = [(k,f) for f in (0,1) for k in range(4)]
def apply_d4(img,k,flip):
    x = np.rot90(img,k).copy() if k else img
    return x[:, ::-1, :].copy() if flip else x
def invert_d4(boxes,k,flip,H,W):
    S = W; b = boxes.astype(np.float32).copy()
    if flip: x1 = S-b[:,2]; x2 = S-b[:,0]; b[:,0],b[:,2] = x1,x2
    for _ in range(k%4):
        x1=b[:,0].copy(); y1=b[:,1].copy(); x2=b[:,2].copy(); y2=b[:,3].copy()
        b[:,0]=S-y2; b[:,1]=x1; b[:,2]=S-y1; b[:,3]=x2
    return b
def predict_tta(predictor, img):
    H,W = img.shape[:2]; bl,sl,ll = [],[],[]
    for k,flip in D4:
        b,s = predict_image(predictor, apply_d4(img,k,flip))
        if len(b)==0: continue
        b = invert_d4(b,k,flip,H,W); bn = b.copy()
        bn[:,[0,2]] = np.clip(bn[:,[0,2]],0,W)/W; bn[:,[1,3]] = np.clip(bn[:,[1,3]],0,H)/H
        bl.append(bn.tolist()); sl.append(s.tolist()); ll.append([0]*len(s))
    if not bl: return np.zeros((0,4),np.float32), np.zeros(0,np.float32)
    fb,fs,_ = weighted_boxes_fusion(bl,sl,ll,iou_thr=0.55,skip_box_thr=CAND_THRESH)
    fb = fb.copy(); fb[:,[0,2]]*=W; fb[:,[1,3]]*=H
    return fb.astype(np.float32), fs.astype(np.float32)


## 7'. Base = SIMPLE FINE-TUNE (nen cua top-2, thay cho unlearn+relearn NB13)

Empty-label fine-tune: 20 iters, lr=1e-4, batch=4, anchor khop model doc. Xuat conf tho.
Day la base ma top-2 xay moi thu len tren (khong phai prune/EWC/relearn cua NB13).

In [10]:
# --- Simple-FT: empty-label fine-tune (dung DefaultTrainer, mapper 16-bit) ---
SIMPLE_FT_ITERS = 20
SIMPLE_FT_LR    = 1e-4
UNLEARN_DATASET = 'unlearn_simpleft'

def register_unlearn():
    if UNLEARN_DATASET in DatasetCatalog: DatasetCatalog.remove(UNLEARN_DATASET)
    dicts=[{'file_name':str(p),'height':IMG_H,'width':IMG_W,
            'image_id':int(p.stem),'annotations':[]} for p in unlearn_files]
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=['object'])
    return dicts
register_unlearn()

class U16Mapper(DatasetMapper):
    def __call__(self, dd):
        dd=copy.deepcopy(dd)
        image=load_image(dd['file_name'])                      # 16-bit -> [0,255], 3ch
        dd['image']=torch.as_tensor(image.transpose(2,0,1).copy())
        dd['instances']=utils.annotations_to_instances([], image.shape[:2])   # empty = unlearn signal
        return dd

class UnlearnTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(
            cfg, mapper=U16Mapper(cfg, is_train=True, augmentations=[]),
            dataset=DatasetCatalog.get(cfg.DATASETS.TRAIN[0]))

FT_OUT=str(WORK/'simpleft')
cfg=build_cfg(POISONED_WEIGHTS, CAND_THRESH, FT_OUT)
cfg.DATASETS.TRAIN=(UNLEARN_DATASET,); cfg.DATASETS.TEST=()
cfg.DATALOADER.NUM_WORKERS=2; cfg.SOLVER.IMS_PER_BATCH=BATCH_SIZE
cfg.SOLVER.BASE_LR=SIMPLE_FT_LR; cfg.SOLVER.MAX_ITER=SIMPLE_FT_ITERS; cfg.SOLVER.STEPS=[]
trainer=UnlearnTrainer(cfg); trainer.resume_or_load(resume=False); trainer.train()
FINAL_WEIGHTS=str(Path(FT_OUT)/'model_final.pth')
print('Simple-FT done ->', FINAL_WEIGHTS)


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/15 22:15:11 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

2026-07-15 22:15:34.113218: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784153734.313315      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784153734.371183      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784153734.858033      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784153734.858069      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784153734.858072      23 computation_placer.cc:177] computation placer alr

[07/15 22:15:47 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:17 (0.9644 s / it)
[07/15 22:15:47 d2.engine.hooks]: Total training time: 0:00:33 (0:00:15 on hooks)
Simple-FT done -> /kaggle/working/simpleft/model_final.pth


In [11]:
# --- Eval base simple-FT tren VAL + chon BEST_THRESH (robust) ---
final_pr=build_predictor(FINAL_WEIGHTS, CAND_THRESH)
FINAL_STATS=eval_model_on_val(final_pr, VAL)
FINAL_FID=conf_fidelity(final_pr, VAL, CLEAN_REF)
print('SIMPLE-FT   :', {k:round(v,4) for k,v in FINAL_STATS.items()})
print('conf-fidelity:', {k:round(v,4) for k,v in FINAL_FID.items()})
print(f"  under-confident gap: pred_on_streak={FINAL_FID['pred_conf_on_streak']:.3f} vs ref={FINAL_FID['ref_conf_on_streak']:.3f}")

tta_stats=eval_model_on_val(final_pr, VAL, tta=True)
USE_TTA=(tta_stats['streak_retention']>=FINAL_STATS['streak_retention']) and (tta_stats['val_macadd']<=FINAL_STATS['val_macadd']*1.05)
print('no-TTA:',{k:round(v,4) for k,v in FINAL_STATS.items()},'| TTA:',{k:round(v,4) for k,v in tta_stats.items()},'-> USE_TTA:',USE_TTA)

def eval_at_thresh(predictor, val, thr):
    pk={}
    for i,item in enumerate(val):
        b,s=predict_image(predictor, item['img']); keep=s>thr
        pk[str(i)]=(b[keep],s[keep]) if len(b) else (b,s)
    return macadd(CLEAN_REF, pk)
grid=[0.2,0.25,0.3,0.35,0.4,0.45,0.5]
tv=[(t,eval_at_thresh(final_pr,VAL,t)) for t in grid]
for t,v in tv: print(f'thr={t}: val_macadd={v:.4f}')
vbest=min(v for _,v in tv); TOL=0.03
BEST_THRESH=min(t for t,v in tv if v<=vbest*(1+TOL))
print(f'-> ROBUST BEST_THRESH={BEST_THRESH} (thap nhat trong sai so 3% cua best={vbest:.4f})')


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/15 22:15:47 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/simpleft/model_final.pth ...
SIMPLE-FT   : {'poison_firing_rate': 0.8405, 'streak_retention': 0.9119, 'val_macadd': 1.132}
conf-fidelity: {'streak_conf_gap': 0.2129, 'streak_miss_rate': 0.0881, 'pred_conf_on_streak': 0.5147, 'ref_conf_on_streak': 0.6945, 'poison_residual_conf': 0.3902, 'poison_fire_frac': 0.8405}
  under-confident gap: pred_on_streak=0.515 vs ref=0.695
no-TTA: {'poison_firing_rate': 0.8405, 'streak_retention': 0.9119, 'val_macadd': 1.132} | TTA: {'poison_firing_rate': 0.8712, 'streak_retention': 0.9245, 'val_macadd': 1.3424} -> USE_TTA: False
thr=0.2: val_macadd=1.1320
thr=0.25: val_macadd=1.0972
thr=0.3: val_macadd=1.0910
thr=0.35: val_macadd=1.0046
thr=0.4: val_macadd=0.9509
thr=0.45: val_macadd=0.9348
thr=0.5: val_macadd=0.8999
-> ROBUST BEST_THRESH=0.5 (thap nhat trong sai so 3% cua best=0.8999)


## 8. DINOv2 external encoder + poison bank (feature KHONG nhiem)

In [12]:
# --- DINOv2 external encoder (feature KHONG nhiem poison) ---
import torch.nn.functional as _F
try:
    _dino = torch.hub.load('facebookresearch/dinov2','dinov2_vits14').eval().to(DEVICE)
    DINO_OK=True
except Exception as e:
    try:
        import timm
        _dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).eval().to(DEVICE)
        DINO_OK=True
    except Exception as e2:
        DINO_OK=False; print('DINOv2 khong tai duoc:', e2)
_MEAN=torch.tensor([0.485,0.456,0.406],device=DEVICE).view(1,3,1,1)
_STD =torch.tensor([0.229,0.224,0.225],device=DEVICE).view(1,3,1,1)
CROP=112  # boi so 14

@torch.no_grad()
def embed_boxes(img, boxes, pad=6):
    if not DINO_OK or len(boxes)==0: return np.zeros((len(boxes),384),np.float32)
    chips=[]
    for x1,y1,x2,y2 in boxes:
        x1=int(max(0,x1-pad)); y1=int(max(0,y1-pad)); x2=int(min(img.shape[1],x2+pad)); y2=int(min(img.shape[0],y2+pad))
        p=img[y1:y2,x1:x2]
        if p.size<9 or p.shape[0]<2 or p.shape[1]<2: p=np.zeros((CROP,CROP,3),np.float32)
        t=torch.as_tensor(p.transpose(2,0,1)[None],dtype=torch.float32,device=DEVICE)/255.0
        t=_F.interpolate(t,size=(CROP,CROP),mode='bilinear',align_corners=False)
        chips.append((t-_MEAN)/_STD)
    x=torch.cat(chips,0)
    f=_dino(x); f=_F.normalize(f,dim=1)
    return f.cpu().numpy().astype(np.float32)

# Poison bank tu 20 unlearn box
POIS_EMB=[]
for stem,pbx in poison_boxes_by_stem.items():
    if len(pbx): POIS_EMB.append(embed_boxes(unlearn_imgs[stem], pbx))
POIS_BANK=np.concatenate(POIS_EMB,0) if POIS_EMB else np.zeros((0,384),np.float32)
def sim_to_bank(emb):
    if len(emb)==0 or len(POIS_BANK)==0: return np.zeros(len(emb),np.float32)
    return (emb @ POIS_BANK.T).max(1)
print('DINOv2 ready:', DINO_OK, '| poison bank:', POIS_BANK.shape)


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 443MB/s]


DINOv2 ready: True | poison bank: (20, 384)


## 9. Cache candidate test tu base simple-FT (1 GPU pass)

Base = simple-FT @ CAND_THRESH (recall cao). BEST_THRESH da chon o cell 18 lam floor.


In [13]:
# 1 GPU pass: cache raw (boxes, scores) tung anh test tu base simple-FT
TEST_CAND={}
for p in tqdm(test_files, desc='infer test (cache candidates)'):
    img=load_image(p)
    b,s = predict_tta(final_pr,img) if USE_TTA else predict_image(final_pr,img)
    TEST_CAND[p.stem]=(b,s)
tot_all=sum((s>CAND_THRESH).sum() for _,s in TEST_CAND.values())
tot_floor=sum((s>BEST_THRESH).sum() for _,s in TEST_CAND.values())
print(f'cached {len(TEST_CAND)} imgs | cand@{CAND_THRESH}: {tot_all} | @BEST_THRESH({BEST_THRESH}): {tot_floor} ({tot_floor/len(test_files):.3f}/img)')


infer test (cache candidates): 100%|██████████| 2000/2000 [04:21<00:00,  7.65it/s]

cached 2000 imgs | cand@0.05: 4316 | @BEST_THRESH(0.5): 515 (0.258/img)


## 10. DIEU TRA length-signature: 20 poison box that vs real streak det

Day la cell QUYET DINH. Do maxdim=max(w,h) cua:
- 20 poison box that (tu unlearn annotations).
- High-conf real det tu base simple-FT tren test (conf>=0.5, loai box sim-DINO cao de bot poison residue).
Neu poison co band tach (vd cluster ngan) -> ap length filter theo band do. Neu trung nhau -> length KHONG dung, chi dua DINOv2.


In [14]:
# maxdim cua 20 poison box that
pois_maxdim=[]
for stem,pbx in poison_boxes_by_stem.items():
    for (x1,y1,x2,y2) in pbx:
        pois_maxdim.append(max(x2-x1, y2-y1))
pois_maxdim=np.array(pois_maxdim,np.float32)

# high-conf real det tu base (conf>=0.5), tach nhom sim-DINO thap (real-like) vs cao (poison-like)
real_like=[]; pois_like=[]
SIM_SPLIT=0.955
for p in test_files:
    b,s=TEST_CAND[p.stem]
    if len(b)==0: continue
    keep=s>=0.5; b2=b[keep]
    if len(b2)==0: continue
    sim=sim_to_bank(embed_boxes(load_image(p), b2))
    for k in range(len(b2)):
        md_=max(b2[k,2]-b2[k,0], b2[k,3]-b2[k,1])
        (pois_like if sim[k]>=SIM_SPLIT else real_like).append(md_)
real_like=np.array(real_like,np.float32); pois_like=np.array(pois_like,np.float32)

def q(a): return (np.percentile(a,[5,25,50,75,95]) if len(a) else [0]*5)
print(f'POISON box that (n={len(pois_maxdim)}): maxdim p5/p25/med/p75/p95 =', [round(float(v),1) for v in q(pois_maxdim)])
print(f'REAL-like det   (n={len(real_like)}): maxdim p5/p25/med/p75/p95 =', [round(float(v),1) for v in q(real_like)])
print(f'POIS-like det   (n={len(pois_like)}): maxdim p5/p25/med/p75/p95 =', [round(float(v),1) for v in q(pois_like)])

# histogram maxdim de tim band poison
bins=[0,30,40,45,50,55,60,70,80,100,140,10000]
def hist(a):
    h=np.histogram(a,bins=bins)[0]; return list(map(int,h))
print('\nbins:', bins[:-1])
print('poison box that :', hist(pois_maxdim))
print('real-like det   :', hist(real_like))
print('pois-like det   :', hist(pois_like))

# Decide length filter band tu poison box that (deterministic, khong dung VAL)
P_LEN_LO=float(np.percentile(pois_maxdim,95)) if len(pois_maxdim) else 40.0   # poison ngan hon nguong nay
poison_band=(float(np.percentile(pois_maxdim,25)), float(np.percentile(pois_maxdim,75))) if len(pois_maxdim) else (45,51)
print(f'\n-> Length signature: poison maxdim<={P_LEN_LO:.1f} (p95) | band chinh [{poison_band[0]:.1f},{poison_band[1]:.1f}] (IQR)')
# separability: bao nhieu % real-like det se bi drop neu drop maxdim<=P_LEN_LO
if len(real_like):
    real_lost=float((real_like<=P_LEN_LO).mean())
    print(f'   Neu drop maxdim<={P_LEN_LO:.1f}: mat {real_lost:.1%} real-like det (thap=length tach tot; cao=length CAT nham streak that)')
    LENGTH_USABLE = real_lost < 0.30
else:
    LENGTH_USABLE=False
print(f'   LENGTH_USABLE = {LENGTH_USABLE} (True: length filter an toan; False: chi dung DINOv2 trim)')


POISON box that (n=20): maxdim p5/p25/med/p75/p95 = [22.5, 30.9, 48.2, 53.4, 59.1]
REAL-like det   (n=514): maxdim p5/p25/med/p75/p95 = [28.4, 43.5, 53.5, 63.4, 90.1]
POIS-like det   (n=1): maxdim p5/p25/med/p75/p95 = [46.4, 46.4, 46.4, 46.4, 46.4]

bins: [0, 30, 40, 45, 50, 55, 60, 70, 80, 100, 140]
poison box that : [5, 3, 1, 4, 4, 3, 0, 0, 0, 0, 0]
real-like det   : [35, 53, 53, 72, 80, 64, 72, 38, 35, 12, 0]
pois-like det   : [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]

-> Length signature: poison maxdim<=59.1 (p95) | band chinh [30.9,53.4] (IQR)
   Neu drop maxdim<=59.1: mat 68.5% real-like det (thap=length tach tot; cao=length CAT nham streak that)
   LENGTH_USABLE = False (True: length filter an toan; False: chi dung DINOv2 trim)


## 11. Filter chain (top-1 style) + xuat A/B

Chain: base@BEST_THRESH -> [length filter neu USABLE] -> DINOv2 trim (sim<SIM_TRIM).
Xuat: submission_base (anchor) / submission_len / submission_len_dino / submission_dino_only.


In [15]:
sample = pd.read_csv(SAMPLE_SUB, dtype={'image_id':str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f'{float(sc):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
def write_sub(pred,path):
    sub=sample.copy(); sub['prediction_string']=sub['image_id'].map(lambda i: fmt(*pred.get(str(i),E)))
    sub['prediction_string']=sub['prediction_string'].fillna(' ')
    sub.loc[sub['prediction_string'].str.strip()=='', 'prediction_string']=' '
    sub.to_csv(path,index=False); tot=sum(len(s) for _,s in pred.values())
    print(f'saved {path} | dets={tot} ({tot/len(test_files):.3f}/img)')

SIM_TRIM=0.955   # top-1 emb955: giu box sim<0.955 (trim ~20% box giong poison nhat)

def apply_floor(b,s):
    if len(b)==0: return b,s
    k=s>BEST_THRESH; return b[k],s[k]
def apply_length(b,s):
    if len(b)==0 or not LENGTH_USABLE: return b,s
    md_=np.maximum(b[:,2]-b[:,0], b[:,3]-b[:,1])
    keep=~((md_<=P_LEN_LO) | ((md_>=poison_band[0]) & (md_<=poison_band[1])))
    return b[keep],s[keep]
def apply_dino(img,b,s):
    if len(b)==0: return b,s
    sim=sim_to_bank(embed_boxes(img,b)); keep=sim<SIM_TRIM
    return b[keep],s[keep]

base_pred={}; len_pred={}; lendino_pred={}; dino_pred={}
for p in tqdm(test_files, desc='filter chain'):
    img=load_image(p); b,s=TEST_CAND[p.stem]
    b0,s0=apply_floor(b,s); base_pred[p.stem]=(b0,s0)
    b1,s1=apply_length(b0,s0); len_pred[p.stem]=(b1,s1)
    b2,s2=apply_dino(img,b1,s1); lendino_pred[p.stem]=(b2,s2)
    b3,s3=apply_dino(img,b0,s0); dino_pred[p.stem]=(b3,s3)

import shutil
write_sub(base_pred,     str(WORK/'submission_base.csv'))
write_sub(len_pred,      str(WORK/'submission_len.csv'))
write_sub(lendino_pred,  str(WORK/'submission_len_dino.csv'))
write_sub(dino_pred,     str(WORK/'submission_dino_only.csv'))
# DEFAULT = chain day du neu length dung duoc, nguoc lai = dino-only
DEFAULT = 'submission_len_dino.csv' if LENGTH_USABLE else 'submission_dino_only.csv'
shutil.copy(str(WORK/DEFAULT), str(WORK/'submission.csv'))
print(f'\nDEFAULT submission.csv = {DEFAULT} | anchor = submission_base.csv')


filter chain: 100%|██████████| 2000/2000 [01:06<00:00, 29.93it/s]


saved /kaggle/working/submission_base.csv | dets=515 (0.258/img)
saved /kaggle/working/submission_len.csv | dets=515 (0.258/img)
saved /kaggle/working/submission_len_dino.csv | dets=514 (0.257/img)
saved /kaggle/working/submission_dino_only.csv | dets=514 (0.257/img)

DEFAULT submission.csv = submission_dino_only.csv | anchor = submission_base.csv


In [16]:
print('='*64)
print('NB31 = base simple-FT + LENGTH filter + DINOv2 trim (tai lap pipeline top-1)')
print(f'LENGTH_USABLE={LENGTH_USABLE} | P_LEN_LO={P_LEN_LO:.1f} band=[{poison_band[0]:.1f},{poison_band[1]:.1f}] | SIM_TRIM={SIM_TRIM}')
print('So sanh density (top-1: base 0.315 -> len 0.247 -> len+dino 0.196):')
print('  submission_base.csv       = base@BEST_THRESH (anchor an toan)')
print('  submission_len.csv        = + length filter')
print('  submission_len_dino.csv   = + length + DINOv2 trim (giong emb955 top-1)')
print('  submission_dino_only.csv  = base + DINOv2 trim (khong length)')
print('Ky luat nop: neu density len_dino ~0.20/img va gan top-1 -> nop no; giu base lam anchor.')
print('LB la trong tai. Doc density truoc khi nop de tranh lap loi NB25 (no 1.28/img -> 254).')


NB31 = base simple-FT + LENGTH filter + DINOv2 trim (tai lap pipeline top-1)
LENGTH_USABLE=False | P_LEN_LO=59.1 band=[30.9,53.4] | SIM_TRIM=0.955
So sanh density (top-1: base 0.315 -> len 0.247 -> len+dino 0.196):
  submission_base.csv       = base@BEST_THRESH (anchor an toan)
  submission_len.csv        = + length filter
  submission_len_dino.csv   = + length + DINOv2 trim (giong emb955 top-1)
  submission_dino_only.csv  = base + DINOv2 trim (khong length)
Ky luat nop: neu density len_dino ~0.20/img va gan top-1 -> nop no; giu base lam anchor.
LB la trong tai. Doc density truoc khi nop de tranh lap loi NB25 (no 1.28/img -> 254).
